In [4]:
import re
import unicodedata
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
import seaborn as sns
import matplotlib.pyplot as plt
import string

1. Base 1 : Narval

In [2]:
path="C:/Users/CD44/Desktop/Narval/narvalentier.xlsx"

In [ ]:
df = pd.read_excel(path) #reading database

In [ ]:
df.shape #cheking the shape

(15535, 40)

In [ ]:
df.head() #visualise the first 5 rows

,id,nom_objet,numero_dossier,demandeur,nom_institution,site,date_entree,date_sortie,numero_inventaire_interne,datation_objet,...,typologie_objet,date_restitution_photos,date_restitution_fiche_restauration,id_dossier,version,numero_fiche,date_modification,nb_reference,matricule,localisation
0,102037,chaussure gauche,00000001,mr chausson,musée de la charentaise,NaN,2019-06-21,NaN,000001,20 ans,...,NaN,NaN,NaN,4426.0,9,000000001,2024-01-09 11:34:14.35,25.0,010393W,NaN
1,102039,chaussure gauche,00000001,mr chausson,musée de la charentaise,NaN,2019-06-21,NaN,000001,20 ans,...,NaN,NaN,NaN,4426.0,2,0000002,2024-01-09 11:34:14.35,NaN,NaN,NaN
2,102038,chaussure gauche,00000001,mr chausson,musée de la charentaise,NaN,2019-06-21,NaN,000001,20 ans,...,NaN,NaN,NaN,4426.0,2,0000002,2024-01-09 11:34:14.35,NaN,NaN,NaN
3,102040,chaussure gauche,00000001,mr chausson,musée de la charentaise,NaN,2019-06-21,NaN,000004,20 ans,...,NaN,NaN,NaN,4426.0,2,000065,2024-01-09 11:34:14.35,NaN,NaN,NaN
4,100050,NaN,D124-747/98,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,3619.0,0,1313156,NaN,NaN,NaN,NaN


In [ ]:
COLUMNS_TO_MERGE = [  #defining the columns to merge in "tout_info" so the study will be done on only one column
    "materiaux_objet",
    "materiaux_complementaire_objet",
    "description_constat_etat",
    "description_traitement",
    "traitement_demande",
    "conseils_conservation"
]

In [7]:
df["tout_info"] = df[COLUMNS_TO_MERGE].apply(
    lambda row: " . ".join(row.dropna().astype(str)),
    axis=1
)

In [8]:
df.head()

,id,nom_objet,numero_dossier,demandeur,nom_institution,site,date_entree,date_sortie,numero_inventaire_interne,datation_objet,...,date_restitution_photos,date_restitution_fiche_restauration,id_dossier,version,numero_fiche,date_modification,nb_reference,matricule,localisation,tout_info
0,102037,chaussure gauche,00000001,mr chausson,musée de la charentaise,NaN,2019-06-21,NaN,000001,20 ans,...,NaN,NaN,4426.0,9,000000001,2024-01-09 11:34:14.35,25.0,010393W,NaN,crêpes
1,102039,chaussure gauche,00000001,mr chausson,musée de la charentaise,NaN,2019-06-21,NaN,000001,20 ans,...,NaN,NaN,4426.0,2,0000002,2024-01-09 11:34:14.35,NaN,NaN,NaN,crêpes
2,102038,chaussure gauche,00000001,mr chausson,musée de la charentaise,NaN,2019-06-21,NaN,000001,20 ans,...,NaN,NaN,4426.0,2,0000002,2024-01-09 11:34:14.35,NaN,NaN,NaN,crêpes
3,102040,chaussure gauche,00000001,mr chausson,musée de la charentaise,NaN,2019-06-21,NaN,000004,20 ans,...,NaN,NaN,4426.0,2,000065,2024-01-09 11:34:14.35,NaN,NaN,NaN,crêpes
4,100050,NaN,D124-747/98,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,3619.0,0,1313156,NaN,NaN,NaN,NaN,


In [14]:
def clean_text(text): 
    if pd.isna(text):
        return text
    
    # Forcer le type string
    text = str(text)
    
    # Minuscules
    text = text.lower()
    
    # Suppression des accents
    text = unicodedata.normalize("NFD", text)
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    
    # Suppression ponctuation / caractères spéciaux
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    
    # Nettoyage espaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [10]:
df["tout_info"] = df["tout_info"].apply(clean_text)

In [11]:
df["tout_info"]

0        crepes
1        crepes
2        crepes
3        crepes
4              
          ...  
15530          
15531          
15532          
15533          
15534          
Name: tout_info, Length: 15535, dtype: str

In [ ]:
df_cuivre = df[df["tout_info"].str.contains(r"\bcuivre\b|\bcuivreux\b|\bbronze\b|\blaiton\b",
                                          case=False, 
                                          na=False)]
#select only the copper elements

In [15]:
keywords = {
    3: ["chalco","chalconatronite"], 
    2: ["Chlorure","Dégradation","Altération","Corrosion localisée","Corrosion verte","Corrosion active","Produit de corrosion bleu-vert","bleu-vert","Pulvérulent","Pulvérulence","Récurrence cyclique","Corrosion pulvérulente","Sels verts"],  

    1: ["Archéologie","Sculpture","Croyance","Croyances-coutumes","Statuette","Statue","Amulette","Monnaie","Basse époque","Époque ptolémaïque","Antiquité","Égypte","Moyen Orient","Pratique votive","Fonte à la cire perdue","Coulé","Cire perdue","Stabilisé","Oxydes d'argent","oxydation","stabilisation chimique", "Sodium"]}

In [16]:
keywords = {
    k: [clean_text(mot) for mot in liste]
    for k, liste in keywords.items()
}  #apply clean_text to keywords too

In [17]:
keywords

{3: ['chalco', 'chalconatronite'],
 2: ['chlorure',
  'degradation',
  'alteration',
  'corrosion localisee',
  'corrosion verte',
  'corrosion active',
  'produit de corrosion bleu vert',
  'bleu vert',
  'pulverulent',
  'pulverulence',
  'recurrence cyclique',
  'corrosion pulverulente',
  'sels verts'],
 1: ['archeologie',
  'sculpture',
  'croyance',
  'croyances coutumes',
  'statuette',
  'statue',
  'amulette',
  'monnaie',
  'basse epoque',
  'epoque ptolemaique',
  'antiquite',
  'egypte',
  'moyen orient',
  'pratique votive',
  'fonte a la cire perdue',
  'coule',
  'cire perdue',
  'stabilise',
  'oxydes d argent',
  'oxydation',
  'stabilisation chimique',
  'sodium']}

In [18]:
SEUIL = 3  #defining threshold to 3 (may be change later!)

In [19]:
#generate the false words from the original in keywords by a distance of 1
def edits1(word, letters):
    splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]
    
    deletes = [L + R[1:] for L, R in splits if R]
    inserts = [L + c + R for L, R in splits for c in letters]
    replaces = [L + c + R[1:] for L, R in splits if R for c in letters]
    
    return set(deletes + inserts + replaces)

#use mapping to link the correct word of its false variants for the scoring
def expand_with_mapping(keywords):
    
    # alphabet optimisé
    all_words = [w for words in keywords.values() for w in words]
    letters = set("".join(all_words))
    
    mapping = {}
    
    for weight, words in keywords.items():
        for word in words:
            w = word  # déjà normalisé
            
            # ✅ mot canonique
            mapping[w] = (w, weight)
            
            # ✅ variantes
            for v in edits1(w, letters):
                if len(v) > 3 and v.isalpha():
                    mapping[v] = (w, weight)
    
    return mapping


In [20]:
def calculer_score_ngram_canonique(row,colonne, mapping, max_n=3):
    
    text = row[colonne]
    tokens = text.split()
    
    score = 0
    details = []
    used_spans = set()
    seen_canonical = set()  # 🔥 clé ici
    
    for n in range(max_n, 0, -1):
        for i in range(len(tokens) - n + 1):
            
            if any(j in used_spans for j in range(i, i+n)):
                continue
            
            ngram = " ".join(tokens[i:i+n])
            
            if ngram in mapping:
                
                canonical_word, weight = mapping[ngram]
                
                # ✅ on vérifie le mot canonique
                if canonical_word in seen_canonical:
                    continue
                
                score += weight
                details.append(f"'{canonical_word}' (poids {weight})")
                
                seen_canonical.add(canonical_word)
                
                for j in range(i, i+n):
                    used_spans.add(j)
    
    return score, " ; ".join(details)

In [22]:
mapping = expand_with_mapping(keywords)

df_cuivre[["score_chalco", "mots_detectes"]] = df_cuivre.apply(
    lambda row: pd.Series(calculer_score_ngram_canonique(row,"tout_info", mapping)),
    axis=1
)


NameError: name 'df_cuivre' is not defined

In [20]:
# Détection finale
df_cuivre["est_chalco"] = df_cuivre["score_chalco"] >= SEUIL

# Résultats
print(f"Éléments détectés comme 'chalconatronite' : {df_cuivre['est_chalco'].sum()} / {len(df_cuivre)}")
print(f"Non détectés : {(~df_cuivre['est_chalco']).sum()}")
print(df_cuivre[df_cuivre["est_chalco"]][["score_chalco", "mots_detectes"]].head(10))

Éléments détectés comme 'chalconatronite' : 763 / 5941
Non détectés : 5178
    score_chalco                                      mots_detectes
11             3          'sodium' (poids 1) ; 'chlorure' (poids 2)
15             4    'pulverulence' (poids 2) ; 'chlorure' (poids 2)
16             3         'monnaie' (poids 1) ; 'chlorure' (poids 2)
19             3           'coule' (poids 1) ; 'chlorure' (poids 2)
22             3           'coule' (poids 1) ; 'chlorure' (poids 2)
29             4  'coule' (poids 1) ; 'chlorure' (poids 2) ; 'st...
34             3       'chlorure' (poids 2) ; 'stabilise' (poids 1)
37             3           'coule' (poids 1) ; 'chlorure' (poids 2)
38             3       'chlorure' (poids 2) ; 'stabilise' (poids 1)
39             3         'monnaie' (poids 1) ; 'chlorure' (poids 2)


In [21]:
score_counts = df_cuivre["score_chalco"].value_counts().sort_index()
print(score_counts)

score_chalco
0    3095
1    1168
2     915
3     410
4     190
5      98
6      44
7       8
8       8
9       5
Name: count, dtype: int64


In [ ]:
#export
df_cuivre["est_chalco"].to_excel("resultats_detection___analyse_est_chalco.xlsx", index=False)

In [ ]:
#exporting all columns
df_cuivre.to_excel("resultats_detection___analyse3.xlsx", index=False)

In [ ]:
# Export 3 : seulement les VRAI 
df_true = df_cuivre[df_cuivre["est_chalco"] == True]

df_true.to_excel("resultats_chalco_detectes___analyse3.xlsx", index=False)

2.data2: GPLA

In [1]:
path_="C:/Users/CD44/Desktop/GPLA/GPLA_excel_.xlsx"


In [5]:
df_ = pd.read_excel(path_) #reading database

In [6]:
df_.shape #cheking the shape

(4541, 255)

In [7]:
df_.head() #display the first 5 rows

,Numéro d'inventaire,Numéro de dépôt,Numéro d'inventaire du déposant,Autre numéro,Autre numéro_2,Autre numéro_3,Autre numéro_4,Autre numéro_5,Autre numéro_6,Désignation du bien,...,Etat_4,Intégrité_4.1,Intégrité_4_2,Intégrité_4_3,Date du constat_4,Dégradation_4.1,Multimédia_4,Notice créée le,Notice modifiée le,Institution / Département
0,09/ S.INV,NaN,NaN,N° système v6 : 73425,NaN,NaN,NaN,NaN,NaN,fragment non déterminé,...,NaN,NaN,NaN,NaN,NaT,NaN,NaN,lundi 22 juillet 2002 10:15:52,mardi 16 mai 2017 12:54:07,Archéologie militaire/Militaria
1,13/ S.INV,NaN,NaN,N° système v6 : 73429,NaN,NaN,NaN,NaN,NaN,obusier,...,NaN,NaN,NaN,NaN,NaT,NaN,NaN,lundi 22 juillet 2002 10:38:33,jeudi 24 juin 2021 08:26:23,Archéologie militaire/Militaria
2,14/ S.INV,NaN,NaN,N° système v6 : 73430,NaN,NaN,NaN,NaN,NaN,obus (paire),...,NaN,NaN,NaN,NaN,NaT,NaN,NaN,lundi 22 juillet 2002 11:02:32,vendredi 17 octobre 2014 10:47:08,Archéologie militaire/Militaria
3,20/ S.INV,NaN,NaN,N° système v6 : 73437,NaN,NaN,NaN,NaN,NaN,épée de ville,...,NaN,NaN,NaN,NaN,NaT,NaN,NaN,lundi 22 juillet 2002 11:30:05,lundi 27 octobre 2014 10:51:49,Archéologie militaire/Militaria
4,22/ S.INV,NaN,NaN,N° système v6 : 73439,NaN,NaN,NaN,NaN,NaN,fourreau d'épée,...,NaN,NaN,NaN,NaN,NaT,NaN,NaN,lundi 22 juillet 2002 11:34:21,lundi 27 octobre 2014 12:52:37,Archéologie militaire/Militaria


In [8]:
COLUMNS_TO_MERGE_ = [  #defining the columns to merge in "tout_info" so the study will be done on only one column
    "Désignation du bien",
    "Désignation du bien_4",
    "Désignation du bien_2",
    "Matière",
    "Matière_2",
    "Matière_3","Matière_4","Matière_5","Matière_6","Matière_7","Matière_8","Matière_9",
    "Technique","Technique_2","Technique_3","Technique_4","Technique_5","Technique_6","Technique_7","Technique_9","Technique_8","Technique_10","Technique_11",
    "Matière_2_2","Matière_2_3","Matière_2_4",'Matière_2_5',
 'Matière_2_6',
 'Matière_3.1',
 'Matière_3_2',
 'Matière_3_3',
 'Matière_4.1',
 'Matière_4_2',
 'Matière_5.1',
 'Matière_5_2',
 'Matière_6.1',
 'Matière_6_2',
 'Matière_7.1','Technique','Technique_2','Technique_4','Technique_5','Technique_6','Technique_7','Technique_8','Technique_9','Technique_10','Technique_11','Technique_2.1','Technique_2_2','Technique_2_3','Technique_2_4','Technique_2_5','Technique_2_6','Technique_2_7','Technique_3_2','Technique_3_3','Technique_3_4','Technique_4.1','Technique_4_2','Technique_5.1','Technique_5_2','Technique_6.1',
 'Technique_6_2','Technique_3','Technique_3.1','Dégradation',
 'Dégradation_2',
 'Dégradation_3',
 'Dégradation_4',
 'Dégradation_5',
 'Dégradation_6',
 'Dégradation_7',
 'Dégradation_8',
 'Dégradation_2_3',
 'Dégradation_2_4',
 'Dégradation_2_5',
 'Dégradation_2_6',
 'Dégradation_3.1',
 'Dégradation_3_2',
 'Dégradation_3_3',"Intervention requise",'Intervention requise_2',
 'Intervention requise_3',
 'Intervention requise_4',
 'Intervention requise_5',
 'Intervention requise_6',
 'Intervention requise_7',
 'Intervention requise_8',
 'Intervention requise_9',
 'Intervention requise_10',
 'Intervention requise_2.1',
 'Intervention requise_2_2',
 'Intervention requise_2_3',
 'Intervention requise_2_4',
 'Intervention requise_2_5',
 'Intervention requise_2_6',
 'Intervention requise_2_7',
 'Intervention requise_2_8',
 'Intervention requise_2_9',
 'Intervention requise_3.1',
 'Intervention requise_3_2',
 'Intervention requise_3_3',
 'Intervention requise_3_4',
 'Intervention requise_3_5',"Description analytique", "Description analytique_2"
]

In [10]:
df_["tout_info_"] = df_[COLUMNS_TO_MERGE_].apply(
    lambda row: " . ".join(row.dropna().astype(str)),
    axis=1
)

In [11]:
df_.head()

,Numéro d'inventaire,Numéro de dépôt,Numéro d'inventaire du déposant,Autre numéro,Autre numéro_2,Autre numéro_3,Autre numéro_4,Autre numéro_5,Autre numéro_6,Désignation du bien,...,Intégrité_4.1,Intégrité_4_2,Intégrité_4_3,Date du constat_4,Dégradation_4.1,Multimédia_4,Notice créée le,Notice modifiée le,Institution / Département,tout_info_
0,09/ S.INV,NaN,NaN,N° système v6 : 73425,NaN,NaN,NaN,NaN,NaN,fragment non déterminé,...,NaN,NaN,NaN,NaT,NaN,NaN,lundi 22 juillet 2002 10:15:52,mardi 16 mai 2017 12:54:07,Archéologie militaire/Militaria,fragment non déterminé . cuivre jaune . liège ...
1,13/ S.INV,NaN,NaN,N° système v6 : 73429,NaN,NaN,NaN,NaN,NaN,obusier,...,NaN,NaN,NaN,NaT,NaN,NaN,lundi 22 juillet 2002 10:38:33,jeudi 24 juin 2021 08:26:23,Archéologie militaire/Militaria,obusier . bronze . bois . corrosion : traces ....
2,14/ S.INV,NaN,NaN,N° système v6 : 73430,NaN,NaN,NaN,NaN,NaN,obus (paire),...,NaN,NaN,NaN,NaT,NaN,NaN,lundi 22 juillet 2002 11:02:32,vendredi 17 octobre 2014 10:47:08,Archéologie militaire/Militaria,obus (paire) . cuivre . forme cylindrique Mêm...
3,20/ S.INV,NaN,NaN,N° système v6 : 73437,NaN,NaN,NaN,NaN,NaN,épée de ville,...,NaN,NaN,NaN,NaT,NaN,NaN,lundi 22 juillet 2002 11:30:05,lundi 27 octobre 2014 10:51:49,Archéologie militaire/Militaria,épée de ville . alliage ferreux . or . . Parti...
4,22/ S.INV,NaN,NaN,N° système v6 : 73439,NaN,NaN,NaN,NaN,NaN,fourreau d'épée,...,NaN,NaN,NaN,NaT,NaN,NaN,lundi 22 juillet 2002 11:34:21,lundi 27 octobre 2014 12:52:37,Archéologie militaire/Militaria,fourreau d'épée . cuir . . Garnitures de laito...


In [23]:
df_["tout_info_"] = df_["tout_info_"].apply(clean_text)

In [24]:
df_["tout_info_"]

0       fragment non determine cuivre jaune liege piec...
1       obusier bronze bois corrosion traces depot art...
2       obus paire cuivre forme cylindrique meme dimen...
3       epee de ville alliage ferreux or partiellement...
4       fourreau d epee cuir garnitures de laiton bout...
                              ...                        
4536    figurine minerve en course alliage cuivreux mo...
4537    epee alliage ferreux laiton forge forge oxydat...
4538    poignard alliage cuivreux usure oxydation cass...
4539    manche de couteau alliage cuivreux moule moule...
4540    instrument de chirurgie alliage cuivreux moule...
Name: tout_info_, Length: 4541, dtype: str

In [26]:
df_cuivre_ = df_[df_["tout_info_"].str.contains(r"\bcuivre\b|\bcuivreux\b|\bbronze\b|\blaiton\b",
                                          case=False, 
                                          na=False)]
#select only the copper elements

In [28]:
mapping = expand_with_mapping(keywords)

df_cuivre_[["score_chalco", "mots_detectes"]] = df_cuivre_.apply(
    lambda row: pd.Series(calculer_score_ngram_canonique(row,"tout_info_", mapping)),
    axis=1
)


In [29]:
# Détection finale
df_cuivre_["est_chalco"] = df_cuivre_["score_chalco"] >= SEUIL

# Résultats
print(f"Éléments détectés comme 'chalconatronite' : {df_cuivre_['est_chalco'].sum()} / {len(df_cuivre_)}")
print(f"Non détectés : {(~df_cuivre_['est_chalco']).sum()}")
print(df_cuivre_[df_cuivre_["est_chalco"]][["score_chalco", "mots_detectes"]].head(10))

Éléments détectés comme 'chalconatronite' : 407 / 4523
Non détectés : 4116
     score_chalco                                      mots_detectes
15              3         'sels verts' (poids 2) ; 'coule' (poids 1)
17              3         'sels verts' (poids 2) ; 'coule' (poids 1)
30              3         'sels verts' (poids 2) ; 'coule' (poids 1)
40              3         'sels verts' (poids 2) ; 'coule' (poids 1)
50              3  'corrosion active' (poids 2) ; 'stabilisation ...
61              3         'sels verts' (poids 2) ; 'coule' (poids 1)
62              3         'sels verts' (poids 2) ; 'coule' (poids 1)
113             3         'sels verts' (poids 2) ; 'coule' (poids 1)
175             3         'sels verts' (poids 2) ; 'coule' (poids 1)
200             3         'sels verts' (poids 2) ; 'coule' (poids 1)


In [30]:
score_counts_ = df_cuivre_["score_chalco"].value_counts().sort_index()
print(score_counts_)

score_chalco
0     1647
1     2199
2      270
3      297
4       29
5       39
6       35
7        6
10       1
Name: count, dtype: int64


In [32]:
#export
df_cuivre_["est_chalco"].to_excel("resultats_detection___analyse_est_chalco_33.xlsx", index=False)

In [33]:
#exporting all columns
df_cuivre_.to_excel("resultats_detection___analyse_33.xlsx", index=False)

In [34]:
# Export 3 : seulement les VRAI 
df_true_ = df_cuivre_[df_cuivre_["est_chalco"] == True]

df_true_.to_excel("resultats_chalco_detectes___analyse_33.xlsx", index=False)